# Drawing Recognition Model

Train digit + letter + shape recognition. Export to TensorFlow.js for Next.js deployment.

In [ ]:
# Cell 1 — Imports and GPU/MPS Setup
# Configure device BEFORE importing TensorFlow (for Apple Silicon Metal)
import os
import random

# Apple Silicon: enable Metal plugin for GPU acceleration
if os.environ.get("TF_METAL") is None:
    os.environ["TF_METAL"] = "1"

import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import sklearn
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import albumentations as A
from tqdm import tqdm
import json
import time
import tensorflow_datasets as tfds

# --- Device detection and report ---
print("=" * 60)
print("ENVIRONMENT REPORT")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Keras version:     {keras.__version__}")

gpus = tf.config.list_physical_devices("GPU")
cpus = tf.config.list_physical_devices("CPU")
# Check for Metal (Apple) — may show as GPU on Mac
all_devices = tf.config.list_physical_devices()
print(f"\nDevices: {[d.device_type for d in all_devices]}")
if gpus:
    for gpu in gpus:
        print(f"  GPU: {gpu.name}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
else:
    print("  No GPU found — using CPU (or Metal on Apple Silicon)")
print("=" * 60)

# --- Reproducibility: set all seeds to 42 ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
# TF deterministic behavior (may slow training slightly)
tf.config.experimental.enable_op_determinism()

print("Random seeds set to 42. Ready for data loading.")

In [ ]:
# Cell 2 — Data Collection and Loading
# Load MNIST, EMNIST (digits, byclass), and Quick Draw; merge into 72-class dataset

# --- 1. MNIST (70k digits, 28x28) ---
(mnist_x_train, mnist_y_train), (mnist_x_test, mnist_y_test) = keras.datasets.mnist.load_data()
mnist_x = np.concatenate([mnist_x_train, mnist_x_test], axis=0)
mnist_y = np.concatenate([mnist_y_train, mnist_y_test], axis=0)
print("1. MNIST:")
print(f"   Shape: {mnist_x.shape}, labels 0-9, total {len(mnist_y)} samples")
print(f"   Class distribution: {np.bincount(mnist_y)}")

# --- 2. EMNIST Digits (280k digits, 28x28) ---
ds_digits = tfds.load("emnist", name="digits", split="train+test", as_supervised=True)
emnist_digits_x, emnist_digits_y = [], []
for img, label in tqdm(ds_digits, desc="EMNIST Digits"):
    # EMNIST image: (28,28,1), may need transpose for display; we'll preprocess later
    emnist_digits_x.append(np.squeeze(img.numpy()))
    emnist_digits_y.append(label.numpy())
emnist_digits_x = np.stack(emnist_digits_x, axis=0)
emnist_digits_y = np.array(emnist_digits_y, dtype=np.int32)
print("\n2. EMNIST Digits:")
print(f"   Shape: {emnist_digits_x.shape}, labels 0-9, total {len(emnist_digits_y)} samples")
print(f"   Class distribution: {np.bincount(emnist_digits_y)}")

# --- 3. EMNIST ByClass (62 classes: 10 digits + 26 upper + 26 lower) ---
ds_byclass = tfds.load("emnist", name="byclass", split="train+test", as_supervised=True)
emnist_byclass_x, emnist_byclass_y = [], []
for img, label in tqdm(ds_byclass, desc="EMNIST ByClass"):
    emnist_byclass_x.append(np.squeeze(img.numpy()))
    emnist_byclass_y.append(label.numpy())
emnist_byclass_x = np.stack(emnist_byclass_x, axis=0)
emnist_byclass_y = np.array(emnist_byclass_y, dtype=np.int32)
print("\n3. EMNIST ByClass:")
print(f"   Shape: {emnist_byclass_x.shape}, labels 0-61, total {len(emnist_byclass_y)} samples")
print(f"   Classes 0-9 (digits): {np.sum((emnist_byclass_y >= 0) & (emnist_byclass_y < 10))}")
print(f"   Classes 10-35 (upper): {np.sum((emnist_byclass_y >= 10) & (emnist_byclass_y < 36))}")
print(f"   Classes 36-61 (lower): {np.sum((emnist_byclass_y >= 36) & (emnist_byclass_y < 62))}")

# --- 4. Quick Draw: 10 shape classes, 10k samples each (28x28) ---
# Use classes that exist in Quick Draw: circle, square, triangle, star, line, zigzag, hexagon, diamond, sun, moon
QUICKDRAW_SHAPES = ["circle", "square", "triangle", "star", "line", "zigzag", "hexagon", "diamond", "sun", "moon"]
SAMPLES_PER_SHAPE = 10_000

qd_info = tfds.builder("quickdraw_bitmap").info
label_names = qd_info.features["label"].names
# Pick 10 class indices: prefer our shape names, then fill from label_names
chosen_indices = []
for s in QUICKDRAW_SHAPES:
    if s in label_names and len(chosen_indices) < 10:
        chosen_indices.append(label_names.index(s))
while len(chosen_indices) < 10:
    for name in label_names:
        idx = label_names.index(name)
        if idx not in chosen_indices:
            chosen_indices.append(idx)
            break
    if len(chosen_indices) >= 10:
        break
chosen_indices = chosen_indices[:10]
qd_label_to_our = {idx: (62 + i) for i, idx in enumerate(chosen_indices)}  # our classes 62-71

ds_qd = tfds.load("quickdraw_bitmap", split="train", as_supervised=True)
count_per_class = {our: 0 for our in range(62, 72)}
qd_x_list, qd_y_list = [], []
for img, label in tqdm(ds_qd, desc="Quick Draw"):
    lab = label.numpy()
    if lab in qd_label_to_our:
        our_label = qd_label_to_our[lab]
        if count_per_class[our_label] < SAMPLES_PER_SHAPE:
            qd_x_list.append(np.squeeze(img.numpy()))
            qd_y_list.append(our_label)
            count_per_class[our_label] += 1
    if all(c >= SAMPLES_PER_SHAPE for c in count_per_class.values()):
        break
quickdraw_x = np.stack(qd_x_list, axis=0)
quickdraw_y = np.array(qd_y_list, dtype=np.int32)
chosen_shape_names = [label_names[i] for i in chosen_indices]
print("\n4. Quick Draw (10 shapes):")
print(f"   Shape: {quickdraw_x.shape}, labels 62-71, total {len(quickdraw_y)} samples")
print(f"   Classes: {chosen_shape_names}")
print(f"   Per-class counts: {count_per_class}")

# --- Merge into unified label space: 0-9 digits, 10-35 upper, 36-61 lower, 62-71 shapes ---
# Digits: merge MNIST + EMNIST Digits + EMNIST ByClass digits (0-9)
# Letters: EMNIST ByClass 10-61 (already 10-35 upper, 36-61 lower)
# Shapes: Quick Draw 62-71

def merge_digits_and_letters():
    x_mnist = mnist_x[..., np.newaxis]  # (N,28,28,1)
    y_mnist = mnist_y
    x_ed = emnist_digits_x[..., np.newaxis]
    y_ed = emnist_digits_y
    # ByClass: digits 0-9 and letters 10-61
    mask_digit = emnist_byclass_y < 10
    mask_letter = emnist_byclass_y >= 10
    x_bc_d = emnist_byclass_x[mask_digit][..., np.newaxis]
    y_bc_d = emnist_byclass_y[mask_digit]
    x_bc_l = emnist_byclass_x[mask_letter][..., np.newaxis]
    y_bc_l = emnist_byclass_y[mask_letter]
    x_digits = np.concatenate([x_mnist, x_ed, x_bc_d], axis=0)
    y_digits = np.concatenate([y_mnist, y_ed, y_bc_d], axis=0)
    x_letters = x_bc_l
    y_letters = y_bc_l
    return x_digits, y_digits, x_letters, y_letters

x_digits, y_digits, x_letters, y_letters = merge_digits_and_letters()
x_shapes = quickdraw_x[..., np.newaxis]
y_shapes = quickdraw_y

X_all = np.concatenate([x_digits, x_letters, x_shapes], axis=0)
y_all = np.concatenate([y_digits, y_letters, y_shapes], axis=0)

# Build label list for later (index -> name)
DIGIT_NAMES = [str(i) for i in range(10)]
UPPER_NAMES = [chr(ord("A") + i) for i in range(26)]
LOWER_NAMES = [chr(ord("a") + i) for i in range(26)]
ALL_LABELS = DIGIT_NAMES + UPPER_NAMES + LOWER_NAMES + chosen_shape_names
assert len(ALL_LABELS) == 72

print("\n" + "=" * 60)
print("FINAL COMBINED DATASET")
print("=" * 60)
print(f"Total samples: {len(X_all)}")
print(f"Total classes: 72 (digits 0-9, uppercase 10-35, lowercase 36-61, shapes 62-71)")
print(f"Image shape: {X_all.shape[1:]}")
print(f"Class distribution (first 15): {np.bincount(y_all, minlength=72)[:15]}")
print("=" * 60)